In [3]:
import pandas as pd

df = pd.read_csv('EURUSD-Hour.csv')


In [4]:


print(df.head())
print(df.info())
print(df.columns)


   2022.11.25  00:00  1.04076  1.04113  1.03961  1.04057  4549  37503
0  2022.11.25  01:00  1.04059  1.04151  1.04034  1.04093  5534  43041
1  2022.11.25  02:00  1.04091  1.04155  1.04082  1.04110  3993  47034
2  2022.11.25  03:00  1.04110  1.04183  1.04045  1.04162  3562  50596
3  2022.11.25  04:00  1.04162  1.04292  1.04161  1.04213  4509  55105
4  2022.11.25  05:00  1.04214  1.04222  1.04073  1.04112  4041  51060
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10438 entries, 0 to 10437
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   2022.11.25  10438 non-null  object 
 1   00:00       10438 non-null  object 
 2   1.04076     10438 non-null  float64
 3   1.04113     10438 non-null  float64
 4   1.03961     10438 non-null  float64
 5   1.04057     10438 non-null  float64
 6   4549        10438 non-null  int64  
 7   37503       10438 non-null  int64  
dtypes: float64(4), int64(2), object(2)
memory usage: 652.5+ K

In [5]:
# Rename columns for clarity
df.columns = [ 'Date', 'Time', 'Open', 'High', 'Low', 'Close', 'TickVolume', 'Bar Volume']


# Combine date and time columns
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)


# Ensure the data is sorted by datetime
df.sort_index(inplace=True)

# Display the updated DataFrame
print(df.head())


                        Open     High      Low    Close  TickVolume  \
Datetime                                                              
2022-11-25 01:00:00  1.04059  1.04151  1.04034  1.04093        5534   
2022-11-25 02:00:00  1.04091  1.04155  1.04082  1.04110        3993   
2022-11-25 03:00:00  1.04110  1.04183  1.04045  1.04162        3562   
2022-11-25 04:00:00  1.04162  1.04292  1.04161  1.04213        4509   
2022-11-25 05:00:00  1.04214  1.04222  1.04073  1.04112        4041   

                     Bar Volume  
Datetime                         
2022-11-25 01:00:00       43041  
2022-11-25 02:00:00       47034  
2022-11-25 03:00:00       50596  
2022-11-25 04:00:00       55105  
2022-11-25 05:00:00       51060  


In [6]:
import ta

def add_indicators_with_crossovers(data):
    # Adding EMA
    data['EMA_4'] = ta.trend.EMAIndicator(data['Close'], window=144).ema_indicator()
    data['EMA_1'] = ta.trend.EMAIndicator(data['Close'], window=21).ema_indicator()
    data['EMA_2'] = ta.trend.EMAIndicator(data['Close'], window=55).ema_indicator()
    data['EMA_3'] = ta.trend.EMAIndicator(data['Close'], window=89).ema_indicator()

    # Adding SMA
    data['SMA_1'] = ta.trend.SMAIndicator(data['Close'], window=50).sma_indicator()
    data['SMA_2'] = ta.trend.SMAIndicator(data['Close'], window=200).sma_indicator()

    # Adding RSI
    data['RSI'] = ta.momentum.RSIIndicator(data['Close'], window=9).rsi()

    # Adding MACD
    macd = ta.trend.MACD(data['Close'], window_slow=26, window_fast=12, window_sign=9)
    data['MACD'] = macd.macd()
    data['MACD_Signal'] = macd.macd_signal()

    # Adding EMA Crossovers
    data['Fast_EMA'] = ta.trend.EMAIndicator(data['Close'], window=13).ema_indicator()
    data['Slow_EMA'] = ta.trend.EMAIndicator(data['Close'], window=48).ema_indicator()
    data['EMA_Crossover'] = data['Fast_EMA'] > data['Slow_EMA']

    # Adding Parabolic SAR
    data['Parabolic_SAR'] = ta.trend.PSARIndicator(data['High'], data['Low'], data['Close']).psar()

    # Adding ADX
    data['ADX'] = ta.trend.ADXIndicator(data['High'], data['Low'], data['Close'], window=14).adx()

    # Adding SuperTrend
    def supertrend(data, period, multiplier):
        st = ta.trend.STCIndicator(close=data['Close'], fillna=True, window_slow=period, window_fast=period//3).stc()
        data[f'SuperTrend_{period}_{multiplier}'] = st

    supertrend(data, period=7, multiplier=3)
    supertrend(data, period=10, multiplier=3)
    supertrend(data, period=20, multiplier=3)

    # Adding Bollinger Bands
    bb = ta.volatility.BollingerBands(close=data['Close'], window=20, window_dev=2)
    data['BB_High'] = bb.bollinger_hband()
    data['BB_Mid'] = bb.bollinger_mavg()
    data['BB_Low'] = bb.bollinger_lband()

    return data

df = add_indicators_with_crossovers(df)


/home/oliverhoffmann/.cache/pypoetry/virtualenvs/qlearning-irBwTS1y-py3.12/lib/python3.12/site-packages/ta/trend.py:1030: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  self._psar[i] = high2


In [7]:
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv
import matplotlib.pyplot as plt
import os

class TradingEnv(gym.Env):
    def __init__(self, df, initial_balance=10000, transaction_cost=0.001):
        super(TradingEnv, self).__init__()

        # Normalize the dataframe
        self.df = (df - df.mean()) / df.std()

        self.initial_balance = initial_balance
        self.transaction_cost = transaction_cost

        self.current_step = 0
        self.total_steps = len(df) - 1

        self.current_balance = initial_balance
        self.net_worth = initial_balance
        self.position = 0

        self.equity_curve = []
        self.episode_length = 0

        # Define action and observation space
        self.action_space = spaces.Discrete(3)  # 0 = sell, 1 = hold, 2 = buy
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.df.shape[1],), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.current_balance = self.initial_balance
        self.net_worth = self.initial_balance
        self.position = 0
        self.equity_curve = []
        self.episode_length = 0

        return self.df.iloc[self.current_step].values, {}

    def step(self, action):
        self.episode_length += 1
        self.current_step += 1

        if self.current_step >= self.total_steps or self.net_worth <= 0.5 * self.initial_balance:
            terminated = True
            truncated = False
        else:
            terminated = False
            truncated = False

        current_price = self.df.iloc[self.current_step]['Close']

        prev_net_worth = self.net_worth

        if action == 0:  # Sell
            if self.position > 0:
                sell_value = self.position * current_price
                transaction_fee = sell_value * self.transaction_cost
                self.current_balance += sell_value - transaction_fee
                self.position = 0

        elif action == 2:  # Buy
            if self.position == 0:
                buy_value = self.current_balance
                transaction_fee = buy_value * self.transaction_cost
                self.position = (buy_value - transaction_fee) / current_price
                self.current_balance = 0

        self.net_worth = self.current_balance + self.position * current_price
        self.equity_curve.append(self.net_worth)

        # Reward is the change in net worth
        reward = self.net_worth - prev_net_worth

        obs = self.df.iloc[self.current_step].values
        info = {'net_worth': self.net_worth, 'trades': int(action != 1)}

        return obs, reward, terminated, truncated, info



'''REMOVE THIS LINE later
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.current_balance = self.initial_balance
        self.net_worth = self.initial_balance
        self.position = 0
        self.position_value = 0
        self.episode_rewards = 0
        self.episode_length = 0
        self.actions = []
        self.equity_curve = []
        return self.df.iloc[self.current_step].values, {}
'''


'REMOVE THIS LINE later\n    def reset(self, seed=None, options=None):\n        super().reset(seed=seed)\n        self.current_step = 0\n        self.current_balance = self.initial_balance\n        self.net_worth = self.initial_balance\n        self.position = 0\n        self.position_value = 0\n        self.episode_rewards = 0\n        self.episode_length = 0\n        self.actions = []\n        self.equity_curve = []\n        return self.df.iloc[self.current_step].values, {}\n'

In [9]:

class CustomLoggerCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(CustomLoggerCallback, self).__init__(verbose)
        self.rewards = []
        self.episode_lengths = []
        self.losses = []

    def _on_step(self) -> bool:
        if 'infos' in self.locals:
            info = self.locals['infos'][0]
            if 'episode' in info:
                self.episode_lengths.append(info['episode'].get('l', 0))
                self.rewards.append(info['episode'].get('r', 0))

        # Capture loss directly from the model's policy if available
        if 'loss' in self.locals:
            self.losses.append(self.locals['loss'])

        return True

    def get_logs(self):
        return {
            "rewards": self.rewards,
            "episode_lengths": self.episode_lengths,
            "losses": self.losses
        }


class PeriodicSaveCallback(BaseCallback):
    def __init__(self, save_freq: int, save_path: str, verbose=1):
        super(PeriodicSaveCallback, self).__init__(verbose)
        self.save_freq = save_freq
        self.save_path = save_path

    def _init_callback(self) -> None:
        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self) -> bool:
        if self.n_calls % self.save_freq == 0:
            model_path = os.path.join(self.save_path, f'model_periodic_{self.n_calls}_steps.zip')
            replay_buffer_path = os.path.join(self.save_path, 'replay_buffer.pkl')

            self.model.save(model_path)
            self.model.save_replay_buffer(replay_buffer_path)

            if self.verbose > 1:
                print(f"Saved model to {model_path}")
                print(f"Saved replay buffer to {replay_buffer_path}")
        return True


In [10]:
import torch
torch.cuda.empty_cache()

In [ ]:
'''kill zombie processes manually'''

#!ps aux | grep python

#kill -9 'PID'

In [14]:
# Paths for saving models and logs
model_save_path = "models/periodic"
replay_buffer_path = os.path.join(model_save_path, 'replay_buffer.pkl')

# Create directories for saving models
os.makedirs(model_save_path, exist_ok=True)

# Check if there is a previously saved model
latest_checkpoint = max(
    [f for f in os.listdir(model_save_path) if f.endswith('.zip')],
    default=None
)


# Remove rows with NaNs
df.dropna(inplace=True)

# Check if any NaNs remain
print(df.isna().sum())

from stable_baselines3.common.monitor import Monitor

def make_env():
    def _init():
        env = TradingEnv(df)
        env = Monitor(env)  # Wrap the environment with Monitor
        return env
    return _init

env = SubprocVecEnv([make_env() for _ in range(4)])



'''REMOVE'''
# Create and wrap the environment
#env = TradingEnv(df)
#env = Monitor(env)

# Initialize the RL agent
#env = SubprocVecEnv([lambda: TradingEnv(df) for _ in range(64)])

# Load existing model if available, otherwise initialize a new one
if latest_checkpoint:
    print(f"Resuming from checkpoint: {latest_checkpoint}")
    model = DQN.load(os.path.join(model_save_path, latest_checkpoint), env=env, device='cuda')
    
    # Load replay buffer if it exists
    if os.path.exists(replay_buffer_path):
        model.load_replay_buffer(replay_buffer_path)
        print(f"Loaded replay buffer from {replay_buffer_path}")
else:
    model = DQN(
        'MlpPolicy', env,
        batch_size=64,
        learning_rate=0.00005,
        buffer_size=2_000_000,
        verbose=1,
        device='cuda'
    )


model = DQN('MlpPolicy', env, batch_size=64, learning_rate=0.00005, buffer_size=2_000_000, verbose=1, device='cuda')

#model = DQN('MlpPolicy', env, verbose=1, device='cpu')

# Initialize callbacks
logger_callback = CustomLoggerCallback()
eval_callback = EvalCallback(
    env,
    best_model_save_path="models/",
    log_path="logs/",
    eval_freq=10000,
    deterministic=True,
    render=False
)
periodic_save_callback = PeriodicSaveCallback(save_freq=100000, save_path=model_save_path)

callbacks = [logger_callback, eval_callback, periodic_save_callback]

try:
    model.learn(total_timesteps=20_000_000, callback=callbacks)
except KeyboardInterrupt:
    print("Training interrupted. Saving model...")
    model.save("models/interrupted_model")
finally:
    # Clean up resources
    env.close()
    torch.cuda.empty_cache()



# Retrieve and plot the logged data
logs = logger_callback.get_logs()


Open               0
High               0
Low                0
Close              0
TickVolume         0
Bar Volume         0
EMA_4              0
EMA_1              0
EMA_2              0
EMA_3              0
SMA_1              0
SMA_2              0
RSI                0
MACD               0
MACD_Signal        0
Fast_EMA           0
Slow_EMA           0
EMA_Crossover      0
Parabolic_SAR      0
ADX                0
SuperTrend_7_3     0
SuperTrend_10_3    0
SuperTrend_20_3    0
BB_High            0
BB_Mid             0
BB_Low             0
dtype: int64
Using cuda device
Using cuda device
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 114      |
|    ep_rew_mean      | -5.1e+03 |
|    exploration_rate | 1        |
| time/               |          |
|    episodes         | 4        |
|    fps              | 5705     |
|    time_elapsed     | 0        |
|    total_timesteps  | 524      |
| train/              |          |
|    learning_rate  

In [ ]:

# Plotting results
plt.figure(figsize=(14, 15))

# Plot rewards
plt.subplot(3, 1, 1)
plt.plot(logs["rewards"], label='Rewards')
plt.title('Episode Rewards')
plt.xlabel('Episodes')
plt.ylabel('Reward')

# Plot episode lengths
plt.subplot(3, 1, 2)
plt.plot(logs["episode_lengths"], label='Episode Lengths')
plt.title('Episode Lengths')
plt.xlabel('Episodes')
plt.ylabel('Length')

# Plot losses
if logs["losses"]:
    plt.subplot(3, 1, 3)
    plt.plot(logs["losses"], label='Losses')
    plt.title('Training Losses')
    plt.xlabel('Steps')
    plt.ylabel('Loss')

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 7))

# Plot equity curve
plt.subplot(3, 1, 1)
plt.plot(env.equity_curve, label='Equity Curve')
plt.title('Equity Curve')
plt.xlabel('Time Steps')
plt.ylabel('Equity')

# Plot buy/sell signals
plt.subplot(3, 1, 2)
plt.plot(df.index, df['Close'], label='Price', color='blue')

buy_signals = [i for i, action in enumerate(env.actions) if action == 2]
sell_signals = [i for i, action in enumerate(env.actions) if action == 0]

buy_dates = df.index[buy_signals]
sell_dates = df.index[sell_signals]

plt.scatter(buy_dates, df.loc[buy_dates, 'Close'], marker='^', color='g', label='Buy Signal')
plt.scatter(sell_dates, df.loc[sell_dates, 'Close'], marker='v', color='r', label='Sell Signal')

plt.title('Buy/Sell Signals')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()

# Plot loss
plt.subplot(3, 1, 3)
# plt.plot(custom_callback.losses, label='Loss')  # If you have loss data
plt.title('Loss')
plt.xlabel('Episodes')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()
